# GRU

### 서울

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0139/0.0341 | R2: 0.9843 | MAE: 4,516 | RMSE: 7,120 | MAPE: 4.50% | MdAPE: 3.48% | RMSLE: 0.0608
Epoch  20 | Loss(T/V): 0.0132/0.0367 | R2: 0.9831 | MAE: 4,577 | RMSE: 7,392 | MAPE: 4.47% | MdAPE: 3.45% | RMSLE: 0.0613
Epoch  30 | Loss(T/V): 0.0124/0.0350 | R2: 0.9839 | MAE: 4,535 | RMSE: 7,208 | MAPE: 4.45% | MdAPE: 3.48% | RMSLE: 0.0610
Epoch  40 | Loss(T/V): 0.0124/0.0333 | R2: 0.9847 | MAE: 4,435 | RMSE: 7,032 | MAPE: 4.39% | MdAPE: 3.37% | RMSLE: 0.0605
Epoch  50 | Loss(T/V): 0.0121/0.0343 | R2: 0.9843 | MAE: 4,502 | RMSE: 7,134 | MAPE: 4.44% | MdAPE: 3.41% | RMSLE: 0.0608
------------------------------
FINAL TEST RESULT: R2: 0.9550 | MAE: 8,999 | RMSE: 13,666 | MAPE: 8.03% | MdAPE: 5.97% | RMSLE: 0.1078


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0073/0.0130 | R2: 0.9938 | MAE: 2,842 | RMSE: 5,100 | MAPE: 2.46% | MdAPE: 1.66% | RMSLE: 0.0397
Epoch  20 | Loss(T/V): 0.0067/0.0148 | R2: 0.9929 | MAE: 3,345 | RMSE: 5,436 | MAPE: 2.93% | MdAPE: 2.17% | RMSLE: 0.0433
Epoch  30 | Loss(T/V): 0.0063/0.0154 | R2: 0.9927 | MAE: 3,378 | RMSE: 5,525 | MAPE: 3.01% | MdAPE: 2.13% | RMSLE: 0.0442
------------------------------
FINAL TEST RESULT: R2: 0.9874 | MAE: 5,687 | RMSE: 7,488 | MAPE: 5.27% | MdAPE: 4.68% | RMSLE: 0.0618


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0074/0.0141 | R2: 0.9932 | MAE: 2,980 | RMSE: 5,330 | MAPE: 2.51% | MdAPE: 1.73% | RMSLE: 0.0400
Epoch  20 | Loss(T/V): 0.0068/0.0144 | R2: 0.9931 | MAE: 3,186 | RMSE: 5,359 | MAPE: 2.73% | MdAPE: 2.00% | RMSLE: 0.0416
Epoch  30 | Loss(T/V): 0.0064/0.0157 | R2: 0.9925 | MAE: 3,177 | RMSE: 5,591 | MAPE: 2.67% | MdAPE: 1.85% | RMSLE: 0.0414
------------------------------
FINAL TEST RESULT: R2: 0.9883 | MAE: 5,315 | RMSE: 7,217 | MAPE: 4.93% | MdAPE: 4.27% | RMSLE: 0.0591


### 부산

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0194/0.0390 | R2: 0.9779 | MAE: 1,756 | RMSE: 3,004 | MAPE: 6.28% | MdAPE: 4.68% | RMSLE: 0.0848
Epoch  20 | Loss(T/V): 0.0178/0.0384 | R2: 0.9783 | MAE: 1,699 | RMSE: 2,977 | MAPE: 5.62% | MdAPE: 4.32% | RMSLE: 0.0753
Epoch  30 | Loss(T/V): 0.0168/0.0302 | R2: 0.9830 | MAE: 1,564 | RMSE: 2,634 | MAPE: 5.31% | MdAPE: 3.96% | RMSLE: 0.0716
------------------------------
FINAL TEST RESULT: R2: 0.9297 | MAE: 3,978 | RMSE: 7,093 | MAPE: 10.44% | MdAPE: 7.74% | RMSLE: 0.1450


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0142/0.0152 | R2: 0.9956 | MAE: 1,383 | RMSE: 2,008 | MAPE: 3.63% | MdAPE: 3.15% | RMSLE: 0.0451
Epoch  20 | Loss(T/V): 0.0136/0.0278 | R2: 0.9920 | MAE: 1,985 | RMSE: 2,706 | MAPE: 5.11% | MdAPE: 4.64% | RMSLE: 0.0601
Epoch  30 | Loss(T/V): 0.0126/0.0196 | R2: 0.9943 | MAE: 1,669 | RMSE: 2,275 | MAPE: 4.56% | MdAPE: 4.01% | RMSLE: 0.0549
------------------------------
FINAL TEST RESULT: R2: 0.9800 | MAE: 3,347 | RMSE: 4,337 | MAPE: 10.70% | MdAPE: 8.55% | RMSLE: 0.1233


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0142/0.0163 | R2: 0.9950 | MAE: 1,466 | RMSE: 2,083 | MAPE: 3.82% | MdAPE: 3.31% | RMSLE: 0.0472
Epoch  20 | Loss(T/V): 0.0136/0.0288 | R2: 0.9912 | MAE: 2,016 | RMSE: 2,758 | MAPE: 5.22% | MdAPE: 4.76% | RMSLE: 0.0610
Epoch  30 | Loss(T/V): 0.0126/0.0209 | R2: 0.9936 | MAE: 1,741 | RMSE: 2,353 | MAPE: 4.77% | MdAPE: 4.29% | RMSLE: 0.0564
------------------------------
FINAL TEST RESULT: R2: 0.9807 | MAE: 3,196 | RMSE: 4,226 | MAPE: 10.13% | MdAPE: 8.10% | RMSLE: 0.1185


### 대구

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0135/0.0231 | R2: 0.9837 | MAE: 998 | RMSE: 1,719 | MAPE: 4.27% | MdAPE: 3.06% | RMSLE: 0.0611
Epoch  20 | Loss(T/V): 0.0125/0.0212 | R2: 0.9851 | MAE: 968 | RMSE: 1,645 | MAPE: 3.91% | MdAPE: 2.88% | RMSLE: 0.0558
------------------------------
FINAL TEST RESULT: R2: 0.9543 | MAE: 1,902 | RMSE: 2,971 | MAPE: 7.49% | MdAPE: 5.33% | RMSLE: 0.1009


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    df = pd.read_csv('sido_대구광역시.csv', encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0103/0.0052 | R2: 0.9970 | MAE: 530 | RMSE: 884 | MAPE: 2.00% | MdAPE: 1.39% | RMSLE: 0.0289
Epoch  20 | Loss(T/V): 0.0099/0.0063 | R2: 0.9964 | MAE: 704 | RMSE: 971 | MAPE: 2.45% | MdAPE: 2.03% | RMSLE: 0.0311
Epoch  30 | Loss(T/V): 0.0093/0.0068 | R2: 0.9961 | MAE: 772 | RMSE: 1,007 | MAPE: 2.69% | MdAPE: 2.36% | RMSLE: 0.0326
Epoch  40 | Loss(T/V): 0.0089/0.0048 | R2: 0.9973 | MAE: 648 | RMSE: 847 | MAPE: 2.27% | MdAPE: 1.92% | RMSLE: 0.0280
------------------------------
FINAL TEST RESULT: R2: 0.9871 | MAE: 1,188 | RMSE: 1,740 | MAPE: 5.26% | MdAPE: 3.61% | RMSLE: 0.0705


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0103/0.0051 | R2: 0.9969 | MAE: 573 | RMSE: 875 | MAPE: 2.16% | MdAPE: 1.53% | RMSLE: 0.0308
Epoch  20 | Loss(T/V): 0.0099/0.0066 | R2: 0.9961 | MAE: 738 | RMSE: 989 | MAPE: 2.62% | MdAPE: 2.19% | RMSLE: 0.0330
Epoch  30 | Loss(T/V): 0.0090/0.0057 | R2: 0.9966 | MAE: 702 | RMSE: 920 | MAPE: 2.56% | MdAPE: 2.17% | RMSLE: 0.0318
------------------------------
FINAL TEST RESULT: R2: 0.9884 | MAE: 1,076 | RMSE: 1,639 | MAPE: 4.58% | MdAPE: 2.90% | RMSLE: 0.0639


### 대전

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0167/0.0611 | R2: 0.9757 | MAE: 1,653 | RMSE: 2,729 | MAPE: 5.77% | MdAPE: 4.79% | RMSLE: 0.0768
Epoch  20 | Loss(T/V): 0.0167/0.0579 | R2: 0.9769 | MAE: 1,565 | RMSE: 2,659 | MAPE: 5.28% | MdAPE: 4.15% | RMSLE: 0.0721
Epoch  30 | Loss(T/V): 0.0159/0.0547 | R2: 0.9781 | MAE: 1,626 | RMSE: 2,588 | MAPE: 6.04% | MdAPE: 4.98% | RMSLE: 0.0823
Epoch  40 | Loss(T/V): 0.0161/0.0503 | R2: 0.9798 | MAE: 1,498 | RMSE: 2,487 | MAPE: 5.18% | MdAPE: 3.97% | RMSLE: 0.0706
Epoch  50 | Loss(T/V): 0.0152/0.0509 | R2: 0.9795 | MAE: 1,536 | RMSE: 2,505 | MAPE: 5.32% | MdAPE: 4.22% | RMSLE: 0.0724
------------------------------
FINAL TEST RESULT: R2: 0.9353 | MAE: 3,375 | RMSE: 5,089 | MAPE: 10.22% | MdAPE: 7.76% | RMSLE: 0.1316


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0127/0.0114 | R2: 0.9961 | MAE: 1,033 | RMSE: 1,432 | MAPE: 2.90% | MdAPE: 2.63% | RMSLE: 0.0352
Epoch  20 | Loss(T/V): 0.0119/0.0141 | R2: 0.9951 | MAE: 1,113 | RMSE: 1,593 | MAPE: 2.87% | MdAPE: 2.59% | RMSLE: 0.0351
Epoch  30 | Loss(T/V): 0.0117/0.0134 | R2: 0.9954 | MAE: 1,105 | RMSE: 1,552 | MAPE: 2.91% | MdAPE: 2.62% | RMSLE: 0.0350
------------------------------
FINAL TEST RESULT: R2: 0.9854 | MAE: 1,924 | RMSE: 2,640 | MAPE: 5.75% | MdAPE: 4.53% | RMSLE: 0.0721


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0127/0.0162 | R2: 0.9941 | MAE: 1,274 | RMSE: 1,710 | MAPE: 3.57% | MdAPE: 3.38% | RMSLE: 0.0414
Epoch  20 | Loss(T/V): 0.0119/0.0192 | R2: 0.9930 | MAE: 1,324 | RMSE: 1,861 | MAPE: 3.44% | MdAPE: 3.19% | RMSLE: 0.0405
Epoch  30 | Loss(T/V): 0.0118/0.0185 | R2: 0.9933 | MAE: 1,343 | RMSE: 1,825 | MAPE: 3.62% | MdAPE: 3.36% | RMSLE: 0.0420
------------------------------
FINAL TEST RESULT: R2: 0.9854 | MAE: 1,879 | RMSE: 2,608 | MAPE: 5.66% | MdAPE: 4.37% | RMSLE: 0.0714


### 광주

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0181/0.0253 | R2: 0.9846 | MAE: 961 | RMSE: 1,453 | MAPE: 4.91% | MdAPE: 3.84% | RMSLE: 0.0667
Epoch  20 | Loss(T/V): 0.0168/0.0241 | R2: 0.9853 | MAE: 918 | RMSE: 1,418 | MAPE: 4.72% | MdAPE: 3.65% | RMSLE: 0.0645
------------------------------
FINAL TEST RESULT: R2: 0.9333 | MAE: 2,278 | RMSE: 3,621 | MAPE: 9.39% | MdAPE: 7.35% | RMSLE: 0.1340


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0116/0.0088 | R2: 0.9959 | MAE: 672 | RMSE: 980 | MAPE: 2.95% | MdAPE: 2.56% | RMSLE: 0.0373
Epoch  20 | Loss(T/V): 0.0115/0.0058 | R2: 0.9973 | MAE: 587 | RMSE: 797 | MAPE: 2.82% | MdAPE: 2.44% | RMSLE: 0.0362
Epoch  30 | Loss(T/V): 0.0109/0.0074 | R2: 0.9966 | MAE: 639 | RMSE: 896 | MAPE: 2.83% | MdAPE: 2.47% | RMSLE: 0.0353
------------------------------
FINAL TEST RESULT: R2: 0.9911 | MAE: 917 | RMSE: 1,504 | MAPE: 3.97% | MdAPE: 2.49% | RMSLE: 0.0590


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의 (GRU)
class MultiRegionGRU(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim)
        self.gru = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, h = self.gru(x_s)
        combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionGRU(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: GRU (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_gru_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_gru_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: GRU (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0116/0.0075 | R2: 0.9964 | MAE: 611 | RMSE: 906 | MAPE: 2.80% | MdAPE: 2.30% | RMSLE: 0.0362
Epoch  20 | Loss(T/V): 0.0110/0.0052 | R2: 0.9975 | MAE: 527 | RMSE: 754 | MAPE: 2.49% | MdAPE: 2.09% | RMSLE: 0.0323
------------------------------
FINAL TEST RESULT: R2: 0.9894 | MAE: 1,025 | RMSE: 1,628 | MAPE: 4.33% | MdAPE: 2.95% | RMSLE: 0.0619
